In [2]:
import json, os, csv
from glob import glob

INPUT_GLOB = "/Users/USER/Documents/Dermatology-Evaluation/DermDetail/qwen_results/*.jsonl"            
OUT_DIR = "/Users/USER/Documents/Dermatology-Evaluation/DermDetail/qwen_layers"
os.makedirs(OUT_DIR, exist_ok=True)

def iter_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

files = sorted(glob(INPUT_GLOB))
version_map = {p: f"v{i+1}" for i, p in enumerate(files)}

fieldnames = ["encounter_id", "version", "image_avg", "text_avg", "sink_avg"]
writers = {}   # layer -> (file_handle, csv_writer)

def get_writer(layer):
    if layer not in writers:
        fh = open(os.path.join(OUT_DIR, f"{layer}.csv"), "w", newline="", encoding="utf-8")
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        writers[layer] = (fh, w)
    return writers[layer][1]

try:
    for path in files:
        version = version_map[path]
        for rec in iter_jsonl(path):
            enc = rec.get("encounter_id", "")
            for layer, vals in (rec.get("attention_ratios_avg") or {}).items():
                if isinstance(vals, dict):
                    get_writer(layer).writerow({
                        "encounter_id": enc,
                        "version": version,
                        "image_avg": vals.get("image"),
                        "text_avg":  vals.get("text"),
                        "sink_avg":  vals.get("sink"),
                    })
finally:
    for fh, _ in writers.values():
        fh.close()

print("Assigned versions:", {os.path.basename(k): v for k, v in version_map.items()})
print("Wrote per-layer CSVs to:", OUT_DIR)

Assigned versions: {'qwen_train_results_v1.jsonl': 'v1', 'qwen_train_results_v2.jsonl': 'v2', 'qwen_train_results_v3.jsonl': 'v3', 'qwen_train_results_v4.jsonl': 'v4', 'qwen_train_results_v5.jsonl': 'v5', 'qwen_train_results_v6.jsonl': 'v6', 'qwen_train_results_v8.jsonl': 'v7'}
Wrote per-layer CSVs to: /Users/USER/Documents/Dermatology-Evaluation/DermDetail/qwen_layers
